This is the notebook where i am trying to learn how can i build a tiny GPT-style character level language model from scratch in pytorch

In [8]:
import torch # PyTorch library for deep learning

with open("input.txt", "r") as file:
    text = file.read()

# print(text)
print("length:",len(text))

length: 115


In [9]:
chars = sorted(list(set(text)))
print("vocabulary size:",len(chars))

vocabulary size: 21


we cant use directly characters because model uses integers so we need to convert every character to integer that called encoding and decoding 
for that we will create two dictionaries stoi(string to integer) and itos(integer to string)

In [10]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [11]:
# encode/decode funtions 
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

# print(decode(encode("hello")))

now lets convert the entire data to a tensor 

In [12]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data)

tensor([ 8,  5, 10, 10, 13,  1, 19, 13, 14, 10,  4,  0,  8,  5, 10, 10, 13,  1,
        16,  8,  5, 14,  5,  0,  8, 13, 19,  1,  2, 14,  5,  1, 20, 13, 17,  0,
         9,  1,  2, 11,  1, 10,  5,  2, 14, 12,  9, 12,  7,  1, 16, 14,  2, 12,
        15,  6, 13, 14, 11,  5, 14, 15,  0, 16, 14,  2, 12, 15,  6, 13, 14, 11,
         5, 14, 15,  1,  2, 14,  5,  1,  9, 12, 16,  5, 14,  5, 15, 16,  9, 12,
         7,  0,  9,  1, 10, 13, 18,  5,  1, 11,  2,  3,  8,  9, 12,  5,  1, 10,
         5,  2, 14, 12,  9, 12,  7])


lets split the data into train and test data set

In [13]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

instead to taking one character at a time we have to train it in chunks 

In [14]:
block_size = 8

In [15]:
#create a training batch 
torch.manual_seed(1337)
batch_size = 4
block_size = 8
def get_batch(split):
    data_source = train_data if split =="train" else val_data

    ix = torch.randint(
        len(data_source)-block_size,(batch_size,)
    )

    x = torch.stack([
        data_source[i:i+block_size] for i in ix
    ])

    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x, y
    

now lets get the train data that we will use for training our first model

In [16]:
xb, yb = get_batch("train")#xb input batch and yb is output means the answer we are expecting from the model
#Input:
# h e l l o _ w o

# Target:
# e l l o _ w o r
# here we can see that target is shifted by one position
print(xb)
print(yb)
print(xb.shape)
print(yb.shape)

tensor([[ 5,  1, 20, 13, 17,  0,  9,  1],
        [ 1, 16,  8,  5, 14,  5,  0,  8],
        [12,  7,  1, 16, 14,  2, 12, 15],
        [ 5,  1, 20, 13, 17,  0,  9,  1]])
tensor([[ 1, 20, 13, 17,  0,  9,  1,  2],
        [16,  8,  5, 14,  5,  0,  8, 13],
        [ 7,  1, 16, 14,  2, 12, 15,  6],
        [ 1, 20, 13, 17,  0,  9,  1,  2]])
torch.Size([4, 8])
torch.Size([4, 8])


the embedding layer we need to add right now we have (B,T) but we want (B,T,C) vectors 

In [17]:
import torch
import torch.nn as nn

# Example
vocab_size = 21
C = 32

token_embedding_table = nn.Embedding(vocab_size, C)

x = token_embedding_table(xb)

print(x.shape)

torch.Size([4, 8, 32])


now we get learnable lookup table 
Suppose initially:
    'h' → [0.1, 0.8, -0.2, ...]
During training, the model makes mistakes.

The loss tells us:

"These parameters need to change."

Gradient descent then updates the embedding vector.

After a lot of training:
    'h' → [0.42, 0.17, -0.81, ...]

The model has learned useful representations.

This is why you shouldn't think of embeddings as fixed definitions.

They are parameters learned by the model.

We've now represented:

h e l l o

as vectors.

But consider:

h e l l o

and:

o l l e h

The same characters are present.

The embedding of h is identical in both cases.

The embedding layer itself doesn't know:

"This h is the first character."

It only knows:

h → vector H

It doesn't know where h occurs in the sequence.

That's a problem.
but right now lets ignore this and see what we got

In [ ]:
# import torch.nn.functional as F
# class BigramlanguageModel(nn.Module):
#     def __init__(self, vocab_size):
#         super().__init__()
#         self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
#     def forward(self,idx,target=None):
#         #idx :(B,T)
#         logits = self.token_embedding_table(idx)
#         #logits :(B,T,V) here V is vector embeddings 
#         if target is None:
#             loss = None
#         else:
#             B,T,V = logits.shape
#             logits = logits.view(B*T,V)
#             target = target.view(B*T) # why we are doing this is dimple are target is (B*T) but are logits are (B,T,V) we want to calculate loss using pytorch loss entropy function but for that it accepts logits 2d and and target 1d
#             loss = F.cross_entropy(logits,target)
#         return logits,loss
#     @torch.no_grad()
#     def generate(self, idx, max_new_tokens):

#         for _ in range(max_new_tokens):

#             logits, loss = self(idx)

#             # logits: (B, T, vocab_size)
#             # Take only the final time step
#             logits = logits[:, -1, :]

#             # logits: (B, vocab_size)

#             probs = F.softmax(logits, dim=-1)

#             # Sample one next character
#             idx_next = torch.multinomial(
#                 probs,
#                 num_samples=1
#             )

#         # idx:       (B, T)
#         # idx_next:  (B, 1)
#         # new idx:   (B, T+1)

#             idx = torch.cat((idx, idx_next), dim=1)

#         return idx

# model = BigramlanguageModel(vocab_size)
# xb,yb = get_batch("train")
# logits , loss = model(xb,yb)#xb inputs and yb target, yb is shifted by one characters 
# print("Input Shape :",xb.shape)
# print("Logits shape ", logits.shape)
# print("Loss          :",loss.item())# for initial loss is estimated to be ln(V) 

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        C = 32

        # What character is this?
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            C
        )

        # Where is this character?
        self.position_embedding_table = nn.Embedding(
            block_size,
            C
        )

        # Convert representation → vocabulary scores
        self.lm_head = nn.Linear(
            C,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token information
        tok_emb = self.token_embedding_table(idx)
        # (B, T, C)

        # Position information
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )
        # (T, C)

        # Combine token + position
        x = tok_emb + pos_emb
        # (B, T, C)

        # Predict next character
        logits = self.lm_head(x)
        # (B, T, V)

        if targets is None:
            loss = None

        else:

            B, T, V = logits.shape

            logits = logits.view(B * T, V)
            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self(idx)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx




Input Shape : torch.Size([4, 8])
Logits shape  torch.Size([32, 21])
Loss          : 3.6076903343200684


In [32]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

for step in range(10000000):
    xb , yb = get_batch("train")
    logits , loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if step%100 == 0:
        print(f"step {step}: loss {loss.item():.4f}")
context = torch.tensor(
    [[stoi["h"]]],
    dtype=torch.long
)

generated = model.generate(
    context,
    max_new_tokens=500
)

print(decode(generated[0].tolist()))

step 0: loss 1.2294
step 100: loss 1.2433
step 200: loss 1.1834
step 300: loss 1.2651
step 400: loss 1.1591
step 500: loss 1.2162
step 600: loss 1.2044
step 700: loss 1.2191
step 800: loss 1.1595
step 900: loss 1.3027
step 1000: loss 1.3140
step 1100: loss 1.3002
step 1200: loss 1.2198
step 1300: loss 1.4085
step 1400: loss 1.1326
step 1500: loss 1.2372
step 1600: loss 1.1863
step 1700: loss 1.2390
step 1800: loss 1.2309
step 1900: loss 1.1520
step 2000: loss 1.3270
step 2100: loss 1.2896
step 2200: loss 1.2843
step 2300: loss 1.2078
step 2400: loss 1.3379
step 2500: loss 1.1971
step 2600: loss 1.2127
step 2700: loss 1.3963
step 2800: loss 1.3818
step 2900: loss 1.4180
step 3000: loss 1.2430
step 3100: loss 1.2724
step 3200: loss 1.3064
step 3300: loss 1.1101
step 3400: loss 1.4682
step 3500: loss 1.3362
step 3600: loss 1.1789
step 3700: loss 1.3743
step 3800: loss 1.3397
step 3900: loss 1.3622
step 4000: loss 1.2096
step 4100: loss 1.3110
step 4200: loss 1.1428
step 4300: loss 1.2564


KeyboardInterrupt: 

In [23]:
context = torch.tensor(
    [[stoi["h"]]],
    dtype=torch.long
)

generated = model.generate(
    context,
    max_new_tokens=500
)

print(decode(generated[0].tolist()))

h

